## **CLEANING LOG**

```
- 300 loaded, 12 prices filled with median
- 8 duplicates removed (Updated to 292 rows)
- zips restored
- dates standardized
- negatives handled as flags for returns
```

In [13]:
import pandas as pd

df = pd.read_csv("messy_sales.csv")

print(df.shape)
print(df.head(10))
print(df.dtypes)

print(f"Pandas version {pd.__version__}")

(300, 6)
   order_id        date   product  price  qty    zip
0      1254  04/06/2026       mug  11.36    4  60614
1      1057  2026-05-30   charger  14.79    3  30303
2      1150  05/06/2026       mug   5.50    1  10001
3      1066  05/30/2026  notebook    NaN    1  98101
4      1114  04/06/2026    webcam  45.59    4  90405
5      1280  04/22/2026   pen set    NaN    4  30303
6      1204  04/18/2026   charger   8.41    2   2134
7      1249  2026-05-15  keyboard  75.04    4  90405
8      1037  2026-05-19       mug   9.31    2  60614
9      1177  2026-05-13       mug  10.11    3  90210
order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object
Pandas version 3.1.0


```
okay, so, as you  can see, for me, it prints date as "str" and not "object", because my pandas version is 3.0+, and not 2.0, so slight issue there. 

anyways, the main issues i can see in the first 10 rows are, well, there are mismatched date formats. then there's missing info for prices in rows 3 and 5. also has 4 digit zip codes for some reason.
```

In [24]:
df.isna().sum()

order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64

In [30]:
fill_value = df["price"].median()
df["price"] = df["price"].fillna(fill_value)
print(df.isna().sum())
print(df.head(10)) # Cleaning Log - median value is 37.53.
print(df.shape)

order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64
   order_id        date   product  price  qty    zip
0      1254  04/06/2026       mug  11.36    4  60614
1      1057  2026-05-30   charger  14.79    3  30303
2      1150  05/06/2026       mug   5.50    1  10001
3      1066  05/30/2026  notebook  37.53    1  98101
4      1114  04/06/2026    webcam  45.59    4  90405
5      1280  04/22/2026   pen set  37.53    4  30303
6      1204  04/18/2026   charger   8.41    2   2134
7      1249  2026-05-15  keyboard  75.04    4  90405
8      1037  2026-05-19       mug   9.31    2  60614
9      1177  2026-05-13       mug  10.11    3  90210
(300, 6)


In [35]:
print("Duplicated rows: ", df.duplicated().sum())

Duplicated rows:  8


In [40]:
df = df.drop_duplicates()
print(df.shape)

(292, 6)


In [42]:
df["zip"] = df["zip"].astype(str).str.zfill(5)
print(df.head(10))

   order_id        date   product  price  qty    zip
0      1254  04/06/2026       mug  11.36    4  60614
1      1057  2026-05-30   charger  14.79    3  30303
2      1150  05/06/2026       mug   5.50    1  10001
3      1066  05/30/2026  notebook  37.53    1  98101
4      1114  04/06/2026    webcam  45.59    4  90405
5      1280  04/22/2026   pen set  37.53    4  30303
6      1204  04/18/2026   charger   8.41    2  02134
7      1249  2026-05-15  keyboard  75.04    4  90405
8      1037  2026-05-19       mug   9.31    2  60614
9      1177  2026-05-13       mug  10.11    3  90210


In [43]:
df["date"] = pd.to_datetime(df["date"], format="mixed")
print(df.dtypes)

order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object


In [46]:
print(df[df["qty"] < 0])

     order_id       date    product  price  qty    zip
202      1140 2026-04-18     webcam  38.69   -5  98101
262      1233 2026-05-09  desk lamp  22.64   -4  10001
297      1025 2026-04-27   keyboard  47.30   -2  02116


In [ ]:
df["possible_return"] = df["qty"] < 0

print(df[df["possible_return"]])

     order_id       date    product  price  qty    zip  possible_return
202      1140 2026-04-18     webcam  38.69   -5  98101             True
262      1233 2026-05-09  desk lamp  22.64   -4  10001             True
297      1025 2026-04-27   keyboard  47.30   -2  02116             True


```
Chose the flagging column method because it was easier than messing with data, good chance that these are returns that the user requested. Since there's only three, it probably might make sense to just straight up remove it, but at the same time, might as well just have the analyst interpret and investigate the data themselves. If I removed them, might have messed with the median. If I corrected them, feels like I'm assuming they're not returns. So overall, I say it's better for a human to interpret the rest, there's only three.
```

In [51]:
df.to_csv("sales_clean.csv", index=False)

### AI USE NOTE
```
Tool: Gemini
Used for: debugging and code syntax, reading lecture transcripts
```